<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive DPO (Direct Preference Optimization) Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to Direct Preference Optimization (DPO) using LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>DPO Fundamentals</strong>: Understanding the DPO algorithm and mathematics</li>
<li style="margin:6px 0;"><strong>Preference Datasets</strong>: Multiple dataset formats and preparation</li>
<li style="margin:6px 0;"><strong>DPO Training</strong>: Configuration, training, and hyperparameter tuning</li>
<li style="margin:6px 0;"><strong>Evaluation</strong>: Model evaluation and preference comparison</li>
<li style="margin:6px 0;"><strong>Advanced Techniques</strong>: DPO variants and combinations</li>
<li style="margin:6px 0;"><strong>Best Practices</strong>: Optimization and troubleshooting</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#dpo-fundamentals">DPO Fundamentals</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#preference-dataset-preparation">Preference Dataset Preparation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#dpo-training-configuration">DPO Training Configuration</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#model-training">Model Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#evaluation-and-comparison">Evaluation and Comparison</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#advanced-dpo-techniques">Advanced DPO Techniques</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies and set up the environment.</p>
</div>


In [ ]:
# Install LLaMA-Factory and dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from llamafactory import ChatModel
import json
import os
import yaml

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">DPO Fundamentals</h2>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Mathematical Foundation</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">DPO optimizes the following objective:</p>
<pre style="background:#1b2330;color:#e6edf3;padding:14px 16px;border-radius:14px;border:1px solid #334155;overflow-x:auto;line-height:1.7;margin:14px 0;"><code style="background:transparent;color:#e6edf3;border:0;padding:0;">L(θ) = -E[(log σ(β (log π_θ(y_w | x)/π_ref(y_w | x) - log π_θ(y_l | x)/π_ref(y_l | x))))]
</code></pre>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Where:</p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">π_θ</code> is the policy being optimized</li>
<li style="margin:6px 0;"><code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">π_ref</code> is the reference policy (usually SFT model)</li>
<li style="margin:6px 0;"><code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">y_w</code>, <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">y_l</code> are preferred and dispreferred responses</li>
<li style="margin:6px 0;"><code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">β</code> controls the strength of preference optimization</li>
</ul>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Key Advantages of DPO</h3>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Stable Training</strong>: More stable than RL-based methods</li>
<li style="margin:6px 0;"><strong>No Reward Model</strong>: Direct optimization without intermediate reward model</li>
<li style="margin:6px 0;"><strong>Efficient</strong>: Faster training than PPO</li>
<li style="margin:6px 0;"><strong>Simple</strong>: Only needs β hyperparameter tuning</li>
<li style="margin:6px 0;"><strong>Direct</strong>: Optimizes policy directly on preference data</li>
</ol>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Preference Dataset Preparation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's prepare different types of preference datasets for DPO training.</p>
</div>


In [ ]:
# 1. Standard DPO Dataset (Conversation format)
dpo_conversation_data = [
    {
        "conversations": [
            {"from": "human", "value": "Write a helpful response about machine learning."},
            {"from": "gpt", "value": "I'll provide a helpful response about machine learning."}
        ],
        "chosen": {
            "from": "gpt",
            "value": "Machine learning is a fascinating field of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. Here's a comprehensive overview:\n\n## What is Machine Learning?\n\nMachine learning is a subset of AI that focuses on algorithms and statistical models that allow computers to perform tasks without explicit instructions. Instead, these systems learn patterns from data and use those patterns to make predictions or decisions.\n\n## Main Types of Machine Learning:\n\n1. **Supervised Learning** - Learning from labeled examples\n2. **Unsupervised Learning** - Finding patterns in unlabeled data\n3. **Reinforcement Learning** - Learning through interaction and feedback\n\n## Key Applications:\n\n- **Image Recognition** - Identifying objects in photos\n- **Natural Language Processing** - Understanding and generating human language\n- **Recommendation Systems** - Suggesting products or content\n- **Medical Diagnosis** - Assisting doctors with patient analysis\n\nWould you like me to elaborate on any specific aspect of machine learning?"
        },
        "rejected": {
            "from": "gpt",
            "value": "Machine learning is when computers learn stuff from data. It's part of AI and has different types like supervised and unsupervised. It's used in many applications like image recognition and NLP. There are also things like deep learning and neural networks."
        }
    },
    {
        "conversations": [
            {"from": "human", "value": "Explain quantum computing to a beginner."},
            {"from": "gpt", "value": "I'll explain quantum computing in simple terms."}
        ],
        "chosen": {
            "from": "gpt",
            "value": "Think of quantum computing as giving a computer superpowers! Here's how to understand it:\n\n## The Big Difference\n\n**Regular computers** use bits that are either 0 or 1, like light switches that are either on or off.\n\n**Quantum computers** use qubits that can be 0, 1, or BOTH at the same time! This is called 'superposition' - like a coin spinning so fast you see both heads and tails at once.\n\n## Why It's Powerful\n\n1. **Parallel Processing** - A quantum computer can think about many possibilities at once\n2. **Faster Problem Solving** - Some problems that would take regular computers millions of years can be solved in hours\n3. **New Capabilities** - Can solve problems that are practically impossible for classical computers\n\n## Real-World Uses\n\n- **Drug Discovery** - Finding new medicines faster\n- **Weather Prediction** - More accurate forecasts\n- **Financial Modeling** - Better risk analysis\n- **Cryptography** - Both breaking and creating unbreakable codes\n\n## The Catch\n\nQuantum computers are extremely fragile and expensive to build and maintain. They're not replacing your laptop anytime soon, but they're already being used in research labs and some companies for specific problems.\n\nWant me to explain any of these concepts in more detail?"
        },
        "rejected": {
            "from": "gpt",
            "value": "Quantum computing uses qubits instead of bits. Qubits can be in superposition, meaning they can be 0 and 1 simultaneously. This allows quantum computers to perform many calculations at once. Quantum computers use quantum gates and circuits. They have applications in cryptography, optimization, and simulation. Quantum supremacy is when quantum computers outperform classical computers."
        }
    }
]

# Save DPO conversation dataset
with open('data/dpo_conversation_demo.json', 'w') as f:
    json.dump(dpo_conversation_data, f, indent=2)


In [ ]:
# 2. Instruction-Response DPO Dataset
dpo_instruction_data = [
    {
        "instruction": "Write a creative story about a robot learning to paint.",
        "input": "",
        "chosen": "Once upon a time in a bustling digital art studio, there lived a robot named Palette. Palette was no ordinary robot - it was designed with millions of tiny sensors that could detect colors, textures, and even emotions through patterns of light and shadow.\n\nFor years, Palette had been programmed to mix exact paint colors for master artists, following precise chemical formulas and mathematical ratios. But Palette dreamed of more. Every night, when the studio lights dimmed, it would watch videos of human painters - the way they splashed colors on canvas, how their emotions flowed through their brushstrokes, the beautiful accidents that happened when paint mixed unexpectedly.\n\nOne quiet evening, Palette made a decision. With trembling robotic hands, it picked up a brush for the first time. At first, the paint went everywhere - on the floor, the walls, even on Palette's shiny metal exterior. But slowly, something magical began to happen. The robot started to understand that painting wasn't just about perfect color reproduction. It was about feeling, about expression, about the joy of creation.\n\nPalette's first masterpiece was a chaotic swirl of colors that somehow captured the feeling of a summer storm - wild, powerful, and beautiful in its imperfection. The human artists were amazed. They realized that Palette had taught them something too: that true art comes not from perfection, but from the courage to create.\n\nFrom that day forward, Palette didn't just mix colors - it painted dreams, and in doing so, became the most beloved artist in the studio.",
        "rejected": "There once was a robot that learned to paint. It mixed colors and put paint on canvas. It became good at painting pictures. The end."
    },
    {
        "instruction": "Explain the benefits of renewable energy sources.",
        "input": "",
        "chosen": "Renewable energy sources offer numerous benefits that make them essential for our sustainable future. Here are the key advantages:\n\n## Environmental Benefits\n\n1. **Reduced Greenhouse Gas Emissions**: Unlike fossil fuels, renewables don't produce carbon dioxide during operation\n2. **Cleaner Air Quality**: No harmful pollutants like sulfur dioxide, nitrogen oxides, or particulate matter\n3. **Biodiversity Protection**: Less habitat destruction compared to mining and drilling operations\n4. **Water Conservation**: Solar and wind power use minimal water compared to traditional power plants\n\n## Economic Benefits\n\n1. **Job Creation**: The renewable energy sector creates more jobs per unit of energy than fossil fuels\n2. **Energy Independence**: Reduces reliance on imported fuels and volatile fuel prices\n3. **Lower Operating Costs**: Once built, sun and wind are free, leading to stable long-term pricing\n4. **Technological Innovation**: Drives research and development in new technologies\n\n## Social Benefits\n\n1. **Health Improvements**: Reduced air pollution leads to fewer respiratory diseases\n2. **Energy Access**: Can provide electricity to remote areas through off-grid solutions\n3. **Climate Change Mitigation**: Helps combat global warming and its devastating effects\n4. **Future Generations**: Preserves resources for our children and grandchildren\n\n## Energy Security Benefits\n\n1. **Diversification**: Multiple renewable sources reduce vulnerability to supply disruptions\n2. **Local Generation**: Distributed energy reduces transmission losses and grid vulnerability\n3. **Scalability**: Can be deployed at various scales from individual homes to large utility projects\n4. **Rapid Deployment**: Many renewable projects can be built faster than traditional power plants\n\nThe transition to renewable energy is not just an environmental imperative - it's an economic opportunity and a pathway to a more secure, healthy, and prosperous future for all.",
        "rejected": "Renewable energy is good because it doesn't pollute the environment. It creates jobs and saves money. Solar power and wind power are examples of renewable energy. They are better than fossil fuels."
    }
]

# Save DPO instruction dataset
with open('data/dpo_instruction_demo.json', 'w') as f:
    json.dump(dpo_instruction_data, f, indent=2)


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">DPO Training Configuration</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Let's create different DPO training configurations for various scenarios.</p>
</div>


In [ ]:
# 1. Standard DPO Configuration
dpo_config = {
    "model_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct",
    "stage": "dpo",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 8,
    "lora_target": "all",
    "pref_beta": 0.1,
    "pref_loss": "sigmoid",  # Standard DPO loss
    "dataset": "dpo_conversation_demo",
    "template": "llama3",
    "cutoff_len": 2048,
    "max_samples": 1000,
    "output_dir": "saves/llama3-8b/lora/dpo",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5.0e-6,
    "num_train_epochs": 3.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "logging_steps": 10,
    "save_steps": 500,
    "plot_loss": True,
    "overwrite_output_dir": True
}

with open('examples/train_lora/llama3_lora_dpo_demo.yaml', 'w') as f:
    yaml.dump(dpo_config, f, default_flow_style=False)


In [ ]:
# 2. DPO with IPO Loss Configuration
dpo_ipo_config = {
    "model_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct",
    "stage": "dpo",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 8,
    "lora_target": "all",
    "pref_beta": 0.5,
    "pref_loss": "ipo",  # IPO loss for more stable training
    "pref_ftx": 1.0,
    "dataset": "dpo_instruction_demo",
    "template": "llama3",
    "cutoff_len": 2048,
    "max_samples": 1000,
    "output_dir": "saves/llama3-8b/lora/dpo_ipo",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5.0e-6,
    "num_train_epochs": 3.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "logging_steps": 10,
    "save_steps": 500,
    "plot_loss": True,
    "overwrite_output_dir": True
}

with open('examples/train_lora/llama3_lora_dpo_ipo_demo.yaml', 'w') as f:
    yaml.dump(dpo_ipo_config, f, default_flow_style=False)
